# Metadata trends + cantonal variation (Bucket-2 family, topic-agnostic core)

Three analyses over structured metadata the generic importer captures for any topic:

- **A — Did Strasbourg get faster?** Proceedings duration (application → judgment/decision)
  by cohort — a policy-relevant question (Protocol 14 entered into force 2010) answered from
  two date columns.
- **B — Violation-rate trends.** Share of merits judgments finding an Art.-8 violation, by
  year band, formation and importance level.
- **C — Cantonal practice variation (CH).** How often Swiss decisions mention specific
  practice instruments (begleitetes Besuchsrecht, Erinnerungskontakte, Kindesanhörung,
  alternierende Obhut) — per canton. **Keyword layer, approximate by design** (stated); the
  per-canton *metadata* split is exact.

A and B are fully topic-agnostic; C's keyword list is topic-adjacent (practice instruments,
not the probe concept) and swappable in one cell.

## A — Strasbourg speed: duration by cohort

In [ ]:
import json
import re
from pathlib import Path
from datetime import datetime
from collections import Counter
import numpy as np
import pandas as pd

DATA_DIR, FIG_DIR, REPORT_DIR = Path("../data"), Path("../figures"), Path("../reports")
echr = json.loads((DATA_DIR / "echr_parental_alienation.json").read_text())

def pdate(s):
    s = (s or "").split()[0] if s else ""
    try:
        return datetime.strptime(s, "%d/%m/%Y").date()
    except (ValueError, IndexError):
        return None

rows = []
for r in echr:
    if (r.get("doctype") or "").upper() == "HECOM":
        continue
    a = pdate(r.get("introductiondate"))
    b = pdate(r.get("judgementdate")) or pdate(r.get("decisiondate"))
    if a and b and b >= a:
        rows.append({"intro_year": a.year, "end_year": b.year,
                     "years": (b - a).days / 365.25, "doctype": r.get("doctype"),
                     "importance": r.get("importance")})
dur = pd.DataFrame(rows)
print(f"computable durations: {len(dur)} (coverage caveat: HUDOC pairs both dates on a subset)\n")

dur["cohort"] = pd.cut(dur.end_year, [1999, 2009, 2014, 2019, 2026],
                       labels=["2000-09", "2010-14", "2015-19", "2020-25"])
print("duration (years, application -> final ruling) by decision cohort:")
print(dur.groupby("cohort", observed=True).years.agg(
    n="size", median="median", mean="mean", p90=lambda s: s.quantile(0.9)).round(1).to_string())
print("\nHow to read : Protocol 14 (2010) aimed at throughput; the cohort medians show whether")
print("              this corpus reflects an acceleration. Coverage bias stated above.")

computable durations: 315 (coverage caveat: HUDOC pairs both dates on a subset)

duration (years, application -> final ruling) by decision cohort:
           n  median  mean  p90
cohort                         
2000-09  100     3.5   3.8  6.7
2010-14   48     2.8   2.7  4.8
2015-19   52     2.5   3.3  6.6
2020-25  115     2.8   3.3  5.4

How to read : Protocol 14 (2010) aimed at throughput; the cohort medians show whether
              this corpus reflects an acceleration. Coverage bias stated above.


## B — Violation-rate trends (merits judgments)

In [ ]:
ext = pd.read_parquet(DATA_DIR / "echr_extracted.parquet")
mer = ext[(ext.genre == "merits") & ext.outcome.isin(["violation", "no violation"])].copy()
mer["year"] = mer.judgment_date.str[:4].astype(float)
mer["band"] = pd.cut(mer.year, [1999, 2012, 2025], labels=["2000-2012", "2013-2025"])
print(f"merits judgments with a clear Art.-8 outcome: {len(mer)}\n")
print("violation share by period:")
print(mer.groupby("band", observed=True).outcome.agg(
    n="size", violation_pct=lambda s: round(100 * (s == "violation").mean(), 1)).to_string())

# by importance level (from raw metadata)
imp = {r.get("itemid"): r.get("importance") for r in echr}
mer["importance"] = mer.id.map(imp)
print("\nviolation share by importance level (1=key case .. 4=routine):")
print(mer.groupby("importance").outcome.agg(
    n="size", violation_pct=lambda s: round(100 * (s == "violation").mean(), 1)).to_string())

def formation(r):
    dc = r.get("documentcollectionid") or ""
    for f in ("GRANDCHAMBER", "CHAMBER", "COMMITTEE"):
        if f in dc:
            return f
    return None
form = {r.get("itemid"): formation(r) for r in echr}
mer["formation"] = mer.id.map(form)
print("\nviolation share by formation:")
print(mer.groupby("formation").outcome.agg(
    n="size", violation_pct=lambda s: round(100 * (s == "violation").mean(), 1)).to_string())
print("\nCaveat: corpus is keyword-matched; shares describe this corpus, not the Court's docket.")

merits judgments with a clear Art.-8 outcome: 566

violation share by period:
             n  violation_pct
band                         
2000-2012  156           64.1
2013-2025  410           69.3

violation share by importance level (1=key case .. 4=routine):
              n  violation_pct
importance                    
1            75           72.0
2            23           65.2
3           248           61.7
4           220           73.6

violation share by formation:
                n  violation_pct
formation                       
CHAMBER       449           62.6
COMMITTEE      78           94.9
GRANDCHAMBER   39           74.4

Caveat: corpus is keyword-matched; shares describe this corpus, not the Court's docket.


## C — Cantonal practice variation (keyword layer, caveated)

In [ ]:
swiss = json.loads((DATA_DIR / "swiss_parental_alienation.json").read_text())

# practice instruments (swappable in one place; \b via regex, case-insensitive)
INSTRUMENTS = {
    "begleitetes Besuchsrecht": r"begleitete[snm]?\s+(besuchsrecht|besuche|kontakt)",
    "Erinnerungskontakte":      r"erinnerungskontakt",
    "Kindesanh\u00f6rung":     r"kindesanh\u00f6rung|anh\u00f6rung des kindes",
    "alternierende Obhut":      r"alternierende[nr]?\s+obhut|wechselmodell",
    "Beistandschaft":           r"beistandschaft",
    "Gutachten":                r"gutachten",
}
COMP = {k: re.compile(v, re.IGNORECASE) for k, v in INSTRUMENTS.items()}

rows = []
for r in swiss:
    txt = r.get("content") or ""
    row = {"canton": r.get("canton"), "year": r.get("year")}
    for k, rx in COMP.items():
        row[k] = bool(rx.search(txt))
    rows.append(row)
cv = pd.DataFrame(rows)
big = cv.groupby("canton").filter(lambda g: len(g) >= 50)   # stable cantons only
tab = (big.groupby("canton")[list(INSTRUMENTS)].mean() * 100).round(0)
tab["n"] = big.groupby("canton").size()
tab = tab.sort_values("n", ascending=False)
print("share of decisions mentioning each practice instrument, per canton (%, n>=50):")
print(tab.to_string())
print("\nCaveat: mention != use; keyword layer is high-recall/approximate; cantonal corpora")
print("differ in court level and publication practice (composition, not just practice, varies).")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
m = tab[list(INSTRUMENTS)].to_numpy()
fig, ax = plt.subplots(figsize=(9, 4.5))
im = ax.imshow(m, cmap="Blues", vmin=0, vmax=100)
ax.set_xticks(range(len(INSTRUMENTS)))
ax.set_xticklabels(list(INSTRUMENTS), rotation=30, ha="right", fontsize=8)
ax.set_yticks(range(len(tab)))
ax.set_yticklabels([f"{i} (n={int(tab.loc[i,'n'])})" for i in tab.index], fontsize=8)
for i in range(m.shape[0]):
    for j in range(m.shape[1]):
        ax.text(j, i, f"{m[i,j]:.0f}", ha="center", va="center", fontsize=7,
                color="white" if m[i, j] > 55 else "#1a3550")
ax.set_title("Swiss cantons: % of decisions mentioning practice instruments")
fig.colorbar(im, shrink=0.8, label="%")
fig.tight_layout()
out = FIG_DIR / "swiss_cantonal_variation.png"
fig.savefig(out, dpi=130); plt.close(fig)
print("wrote", out.name)

share of decisions mentioning each practice instrument, per canton (%, n>=50):
        begleitetes Besuchsrecht  Erinnerungskontakte  Kindesanhörung  alternierende Obhut  Beistandschaft  Gutachten    n
canton                                                                                                                    
ZH                          33.0                  2.0             7.0                 19.0            64.0       55.0  621
CH                          23.0                  2.0            18.0                 12.0            50.0       47.0  514
BL                          29.0                  2.0            11.0                 11.0            71.0       46.0  177
GR                          31.0                  7.0            20.0                 15.0            78.0       60.0  172
BS                          35.0                  2.0            19.0                 14.0            63.0       34.0  136
AG                          31.0                  2.0       

## Report

In [ ]:
lines = ["# Metadata trends + cantonal variation (Bucket-2 family)\n\n"]
coh = dur.groupby("cohort", observed=True).years.median().round(1)
lines.append("## Strasbourg speed (application -> final ruling)\n")
lines.append("- median duration by decision cohort: " +
             "; ".join(f"**{k}: {v}y**" for k, v in coh.items()) +
             f" (n={len(dur)}; coverage caveat: both dates present on a subset).\n\n")
vb = mer.groupby("band", observed=True).outcome.apply(lambda s: round(100*(s=="violation").mean(),1))
lines.append("## Violation-rate trends (merits, clear Art.-8 outcome)\n")
lines.append("- violation share: " + "; ".join(f"**{k}: {v}%**" for k, v in vb.items()) + "\n")
vf = mer.groupby("formation").outcome.apply(lambda s: round(100*(s=="violation").mean(),1))
lines.append("- by formation: " + "; ".join(f"{k} {v}%" for k, v in vf.items()) + "\n\n")
lines.append("## Cantonal practice variation (keyword layer, caveated)\n")
lines.append(f"- {len(tab)} cantons with n>=50 decisions; figure "
             "`figures/swiss_cantonal_variation.png`. Mention != use; cantonal corpora "
             "differ in composition.\n\n")
lines.append("## Reusability note\n"
             "- A and B consume only importer metadata -> run unchanged on any topic corpus.\n"
             "- C's instrument lexicon is swappable in one cell; the per-canton split itself "
             "is exact metadata.\n")
out = REPORT_DIR / "trends_variation_report.md"
out.write_text("".join(lines), encoding="utf-8")
print("wrote", out)

wrote ../reports/trends_variation_report.md
